# Notebook 05 - Pitch Estimation Methods

## Goal
Compare autocorrelation, AMDF, and ZCR-based pitch heuristics.


## Agenda
- Pick one frame
- Estimate F0 by autocorrelation
- Estimate F0 by AMDF
- Estimate ZCR proxy and compare


## Concept and Math

If period is P samples, then F0 = f_s / P.
Autocorrelation finds lag with high periodic similarity.
AMDF finds lag with low average difference.
ZCR is a rough heuristic, not a robust pitch estimator.


In [ ]:
from pathlib import Path
import numpy as np
import librosa as lb
import librosa.display
import matplotlib.pyplot as plt

DATA_ROOT = Path("../dataset")
audio_files = sorted(DATA_ROOT.rglob("*.flac")) + sorted(DATA_ROOT.rglob("*.wav"))
if not audio_files:
    raise FileNotFoundError("No .flac or .wav found under ../dataset")

audio_path = audio_files[0]
print(f"Using: {audio_path}")

wave, sr = lb.load(audio_path, sr=16000, mono=True)
frame = wave[len(wave)//3:len(wave)//3 + 1024]


def f0_autocorr(x, sr, fmin=60, fmax=400):
    x = x - np.mean(x)
    ac = np.correlate(x, x, mode="full")[len(x)-1:]
    lo, hi = int(sr/fmax), int(sr/fmin)
    lag = lo + np.argmax(ac[lo:hi])
    return sr / lag


def f0_amdf(x, sr, fmin=60, fmax=400):
    lo, hi = int(sr/fmax), int(sr/fmin)
    errs = [np.mean(np.abs(x[:-lag] - x[lag:])) for lag in range(lo, hi)]
    lag = lo + int(np.argmin(errs))
    return sr / lag

zcr = lb.feature.zero_crossing_rate(frame, frame_length=len(frame), hop_length=len(frame))[0, 0]
print("autocorr:", f0_autocorr(frame, sr))
print("amdf:", f0_amdf(frame, sr))
print("zcr_proxy:", zcr * sr / 2.0)


## PyTorch Equivalent Snippet
Understand the librosa block first, then map it to this snippet.


In [ ]:
import torchaudio
import torch

wave_t, sr_t = torchaudio.load(str(audio_path))
wave_t = wave_t.mean(dim=0, keepdim=True)
pitch_t = torchaudio.functional.detect_pitch_frequency(wave_t, sr_t)
print("median_pitch:", torch.nanmedian(pitch_t).item())


## Review Checklist
- How does lag become frequency?
- Why does AMDF use argmin while autocorr uses argmax?
- When does ZCR break down?
